# Pruning

In [4]:
import torch
import torch_pruning as tp

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import os
import dotenv

dotenv.load_dotenv()

True

# Upload Gemma-3 270M

In [26]:
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)
device = "cpu"

MODEL_NAME = "google/gemma-3-270m-it"
DTYPE = torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE,).to(device)

print("Model loaded on:", model.device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Model loaded on: cpu


In [31]:
def inference(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=True,
            temperature=0.7
        )

    return inputs, tokenizer.decode(outputs[0], skip_special_tokens=True)

In [48]:
def describe_model(model):
    print(type(model).__name__)
    print(model.model.layers)

In [49]:
tokenized_prompt, answer = inference(model, prompt="Answer in one word what is 1 + 1")
print(answer)

Answer in one word what is 1 + 1 + 1 + 1 + 1 +


# Model structure

In [50]:
describe_model(model)

Gemma3ForCausalLM
ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
      (up_proj): Linear(in_features=640, out_features=2048, bias=False)
      (down_proj): Linear(in_features=2048, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,

In [51]:
print(f"Parameters before pruning: {tp.utils.count_params(model)}")

Parameters before pruning: 268098176


# Pruning experiment #1

Within this experiment, we will try to delete some MLP channels with the smallest L2 norm. Each channel is deleted jointly from the gate, up, and down projections.”

In [84]:
from copy import deepcopy
from torch import nn


def forward_fn(model, input_ids):
    return model(input_ids=input_ids, use_cache=False, return_dict=False)[0]


def prune_blocks(model, ratio=0.25):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.pruner.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [91]:
pruned_model = prune_blocks(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.0.pre_feedforward_layernorm.weight', 'model.layers.2.input_layernorm.weight', 'model.layers.4.post_feedforward_layernorm.weight', 'model.layers.12.pre_feedforward_layernorm.weight', 'model.layers.14.post_feedforward_layernorm.weight', 'model.layers.15.self_attn.q_norm.weight', 'model.layers.3.post_feedforward_layernorm.weight', 'model.layers.10.self_attn.k_norm.weight', 'model.layers.5.self_attn.k_norm.weight', 'model.layers.8.pre_feedforward_layernorm.weight', 'model.layers.12.input_layernorm.weight', 'model.layers.14.post_attention_layernorm.weight', 'model.layers.17.pre_feedforward_layernorm.weight', 'model.layers.15.pre_feedforward_layernorm.weight', 'model.layers.5.input_layernorm.weight', 'model.layers.9.post_feedforward_layernorm.weight', 'model.layers.3.post_attention_layernorm.weight', 'model.layers.4.self_att

In [92]:
print(inference(pruned_model, "Hello, how are you?")[1])

Hello, how are you? ***ign

 AS
 3


In [87]:
print(pruned_model.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=2000, bias=False)
      (up_proj): Linear(in_features=640, out_features=2000, bias=False)
      (down_proj): Linear(in_features=2000, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [93]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model)}")

Parameters after pruning: 260909696


### Simple test (before vs after)

In [94]:
def evaluate_model(model, text, tokenizer):
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        outputs = model(
            **inputs,
            labels=inputs["input_ids"],
            use_cache=False,
        )

    loss = outputs.loss.item()

    return { "loss": loss }

In [95]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 20.359861373901367}


As we can see, deleting the weights with the smallest norm is not a good way to make a model pruning. The loss increased 4 times which is not acceptable.

# Pruning experiment #2

Here, we will try to do the same pruning of MLP but now delete channels with the biggest L2 norm.

In [98]:
def prune_model_2(model, ratio=0.1):
    model = deepcopy(model)

    mlp_roots = {
        proj
        for block in model.model.layers
        for proj in (block.mlp.gate_proj, block.mlp.up_proj, block.mlp.down_proj)
    }

    ignored_layers = [
        module
        for module in model.modules()
        if isinstance(module, nn.Linear) and module not in mlp_roots
    ]

    pruner = tp.MagnitudePruner(
        model=model,
        example_inputs=tokenized_prompt["input_ids"].to(model.device),
        forward_fn=forward_fn,
        importance=tp.importance.GroupMagnitudeImportance(p=2, normalizer=lambda s: -s),
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
        root_module_types=[nn.Linear],
        round_to=8,
    )

    pruner.step()
    return model

In [99]:
pruned_model2 = prune_model_2(model, ratio=0.1)

/Users/mac/PycharmProjects/small-LLM/.venv/lib/python3.14/site-packages/torch_pruning/dependency/graph.py:390: UserWarning: Unwrapped parameters detected: ['model.layers.13.post_feedforward_layernorm.weight', 'model.layers.10.input_layernorm.weight', 'model.layers.15.self_attn.q_norm.weight', 'model.layers.2.pre_feedforward_layernorm.weight', 'model.layers.2.post_feedforward_layernorm.weight', 'model.layers.3.post_feedforward_layernorm.weight', 'model.layers.4.post_feedforward_layernorm.weight', 'model.layers.14.self_attn.k_norm.weight', 'model.layers.15.post_attention_layernorm.weight', 'model.layers.2.post_attention_layernorm.weight', 'model.layers.4.pre_feedforward_layernorm.weight', 'model.layers.6.pre_feedforward_layernorm.weight', 'model.layers.9.self_attn.k_norm.weight', 'model.layers.17.post_feedforward_layernorm.weight', 'model.layers.7.post_attention_layernorm.weight', 'model.layers.14.post_attention_layernorm.weight', 'model.layers.2.self_attn.q_norm.weight', 'model.layers.2

In [103]:
print(inference(pruned_model2, "Hello, how are you?")[1])

Hello, how are you??……?…?…?…?…?…?…?…?


In [104]:
print(pruned_model2.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1840, bias=False)
      (up_proj): Linear(in_features=640, out_features=1840, bias=False)
      (down_proj): Linear(in_features=1840, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [105]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model2)}")

Parameters after pruning: 260909696


### Simple test (before vs after)

In [106]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model2, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 8.798967361450195}


As we can see, the result is much more better, but still the model performs twice worse than before.

# Pruning experiment #3

Here, we will try to delete random channels from MLP.

In [109]:
def prune_model3(model, ratio=0.1, seed=42):
    model = deepcopy(model)

    torch.manual_seed(seed)

    for block in model.model.layers:
        n_prune = int(block.mlp.gate_proj.out_features * ratio)
        idx = torch.randperm(block.mlp.gate_proj.out_features)[:n_prune].tolist()

        tp.prune_linear_out_channels(block.mlp.gate_proj, idx)
        tp.prune_linear_out_channels(block.mlp.up_proj, idx)
        tp.prune_linear_in_channels(block.mlp.down_proj, idx)

    return model

In [111]:
pruned_model3 = prune_model3(model)

In [112]:
print(inference(pruned_model3, "Hello, how are you?")[1])

Hello, how are you?
Is there a chance of rain tomorrow evening?


In [113]:
print(pruned_model3.model.layers)

ModuleList(
  (0-17): 18 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=640, out_features=1024, bias=False)
      (k_proj): Linear(in_features=640, out_features=256, bias=False)
      (v_proj): Linear(in_features=640, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=640, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=640, out_features=1844, bias=False)
      (up_proj): Linear(in_features=640, out_features=1844, bias=False)
      (down_proj): Linear(in_features=1844, out_features=640, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((640,), eps=1e-06)
  )


In [114]:
print(f"Parameters after pruning: {tp.utils.count_params(pruned_model3)}")

Parameters after pruning: 261047936


In [115]:
test_text = """
Artificial intelligence is a field of computer science.
Language models predict tokens based on preceding context.
"""

print("Original:", evaluate_model(model, test_text, tokenizer))
print("Pruned:  ", evaluate_model(pruned_model3, test_text, tokenizer))

Original: {'loss': 4.981411933898926}
Pruned:   {'loss': 5.343407154083252}


That is the best result that we had. 7% of loss after pruning.